In [209]:
"""Currency converter backed by the Frankfurter API (https://frankfurter.dev).

Uses the v2 endpoint `GET /v2/rate/{base}/{quote}`, which returns the rate for
a single currency pair (optionally for a historical `date`).

Requires: requests  (pip install requests)
"""

from datetime import date
import requests
import json
import sqlite3

In [156]:
baseURL = 'https://api.frankfurter.dev/v2/rates'
payload = {"base":"usd", "quotes":"sar,gbp","from":"2026-10-01"}
response = requests.get(f"{baseURL}",params=payload)

# Ensure the request succeeded
response.raise_for_status()


In [157]:
response.json()

[{'date': '2026-10-01', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75428},
 {'date': '2026-10-01', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-02', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75549},
 {'date': '2026-10-02', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-03', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75564},
 {'date': '2026-10-03', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-04', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75568},
 {'date': '2026-10-04', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-05', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75568},
 {'date': '2026-10-05', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-06', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75469},
 {'date': '2026-10-06', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},
 {'date': '2026-10-07', 'base': 'USD', 'quote': 'GBP', 'rate': 0.75467},
 {'date': '2026-10-07', 'base': 'USD', 'quote': 'SAR', 'rate': 3.75},

In [ ]:
class CurrencyConverter():
    TARGET_CURRENCY = "USD"
    Currencies = ['USD', 'EUR', 'INR', 'CAD', 'GBP', 'AUD', 'AED']
    
    def __init__(self, base = TARGET_CURRENCY, quotes = Currencies):
        self.TARGET_CURRENCY = base
        self.Currencies = quotes
        self.db = self.init_database()

    def request_rates(self, date= date.today().strftime("%Y-%m-%d")):
        payload = {"base":self.TARGET_CURRENCY, 
                   # get quote of all currency except the target if its in currencies
                   "quotes": ",".join([x for x in self.Currencies if x != self.TARGET_CURRENCY]), 
                   "date":date}
        return self.request(payload)

    def request_range_rates(self, since):
        payload = {"base":self.TARGET_CURRENCY, 
                   # get quote of all currency except the target if its in currencies
                   "quotes": ",".join([x for x in self.Currencies if x != self.TARGET_CURRENCY]), 
                   "from":since}
        return self.request(payload)
    
    def request(self, payload):
        response = requests.get(baseURL,params=payload)    
        self.save_rates(response.json())
        return response

    def init_database(self):
        db = sqlite3.connect("rates.db")
        db.execute("""CREATE TABLE IF NOT EXISTS rates (
            date TEXT, base TEXT, quote TEXT, rate REAL,
            PRIMARY KEY (date, base, quote))""")
        return db
        
    def save_rates(self, rates: list):
        self.db.executemany("INSERT OR REPLACE INTO rates VALUES (:date,:base,:quote,:rate)",rates)
        self.db.commit()
    
    def get_rate(self, quote, base = TARGET_CURRENCY, date = date.today().strftime("%Y-%m-%d")):
        row = self.db.execute(
            "SELECT rate FROM rates WHERE date=? AND base=? AND quote=?",
            (date, base, quote),
        ).fetchone()
        return row
    
    def convert(self, value: float, quote, base = TARGET_CURRENCY):
        ...

    
    

In [242]:
c = CurrencyConverter()

In [247]:
rates = c.request_rates()


In [248]:
rates.url

'https://api.frankfurter.dev/v2/rates?base=USD&quotes=EUR%2CINR%2CCAD%2CGBP%2CAUD%2CAED&date=2026-10-25'

In [249]:
rates = rates.json()
rates

[{'date': '2026-10-25', 'base': 'USD', 'quote': 'AED', 'rate': 3.6725},
 {'date': '2026-10-25', 'base': 'USD', 'quote': 'AUD', 'rate': 1.425},
 {'date': '2026-10-25', 'base': 'USD', 'quote': 'CAD', 'rate': 1.4006},
 {'date': '2026-10-25', 'base': 'USD', 'quote': 'EUR', 'rate': 0.89145},
 {'date': '2026-10-25', 'base': 'USD', 'quote': 'GBP', 'rate': 0.74689},
 {'date': '2026-10-25', 'base': 'USD', 'quote': 'INR', 'rate': 94.81}]

In [ ]:
row = db.execute(
    "SELECT rate FROM rates WHERE date=? AND base=? AND quote=?",
    ("2024-05-15", "EUR", "USD"),
).fetchone()

[('2026-10-01', 'USD', 'AED', 3.6725),
 ('2026-10-01', 'USD', 'AUD', 1.4382),
 ('2026-10-01', 'USD', 'CAD', 1.4216),
 ('2026-10-01', 'USD', 'EUR', 0.88287),
 ('2026-10-01', 'USD', 'GBP', 0.75428),
 ('2026-10-01', 'USD', 'INR', 95.99),
 ('2026-10-02', 'USD', 'AED', 3.6725),
 ('2026-10-02', 'USD', 'AUD', 1.4401),
 ('2026-10-02', 'USD', 'CAD', 1.4226),
 ('2026-10-02', 'USD', 'EUR', 0.88567)]

In [238]:
row = db.execute(
    "SELECT * FROM rates").fetchall()
row

[('2026-10-01', 'USD', 'AED', 3.6725),
 ('2026-10-01', 'USD', 'AUD', 1.4382),
 ('2026-10-01', 'USD', 'CAD', 1.4216),
 ('2026-10-01', 'USD', 'EUR', 0.88287),
 ('2026-10-01', 'USD', 'GBP', 0.75428),
 ('2026-10-01', 'USD', 'INR', 95.99),
 ('2026-10-02', 'USD', 'AED', 3.6725),
 ('2026-10-02', 'USD', 'AUD', 1.4401),
 ('2026-10-02', 'USD', 'CAD', 1.4226),
 ('2026-10-02', 'USD', 'EUR', 0.88567),
 ('2026-10-02', 'USD', 'GBP', 0.75549),
 ('2026-10-02', 'USD', 'INR', 96.16),
 ('2026-10-03', 'USD', 'AED', 3.6725),
 ('2026-10-03', 'USD', 'AUD', 1.4404),
 ('2026-10-03', 'USD', 'CAD', 1.4228),
 ('2026-10-03', 'USD', 'EUR', 0.88616),
 ('2026-10-03', 'USD', 'GBP', 0.75564),
 ('2026-10-03', 'USD', 'INR', 96.18),
 ('2026-10-04', 'USD', 'AED', 3.6725),
 ('2026-10-04', 'USD', 'AUD', 1.4404),
 ('2026-10-04', 'USD', 'CAD', 1.4229),
 ('2026-10-04', 'USD', 'EUR', 0.88658),
 ('2026-10-04', 'USD', 'GBP', 0.75568),
 ('2026-10-04', 'USD', 'INR', 96.2),
 ('2026-10-05', 'USD', 'AED', 3.6725),
 ('2026-10-05', 'USD',